In [ ]:
import requests
import time
import json
from datetime import datetime, timedelta
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, JSON
import base64

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('default')

class GitHubAPITester:
    
    def __init__(self, token=None):
        self.base_url = "https://api.github.com"
        self.headers = {
            'Accept': 'application/vnd.github.v3+json',
            'User-Agent': 'GitHub-Research/1.0'
        }
        
        if token:
            self.headers['Authorization'] = f'token {token}'
            print("авторизованный доступ")
        else:
            print("неавторизованный доступ")
        
        self.request_count = 0
        self.start_time = time.time()
    
    def _request(self, endpoint, params=None):
        url = f"{self.base_url}{endpoint}"
        self.request_count += 1
        
        try:
            response = requests.get(url, headers=self.headers, params=params)
            
            print(f"\n[{self.request_count}] {endpoint}")
            if params:
                print(f"   Параметры: {params}")
            print(f"   Статус: {response.status_code}")
            print(f"   Осталось запросов: {response.headers.get('X-RateLimit-Remaining', 'N/A')}")
            
            response.raise_for_status()
            return response.json()
            
        except requests.exceptions.RequestException as e:
            print(f"ошибка: {e}")
            return None
    
    def search_repos(self, query, page=1, per_page=30):
        return self._request('/search/repositories', {
            'q': query,
            'page': page,
            'per_page': per_page
        })
    
    def get_repo(self, owner, repo):
        return self._request(f'/repos/{owner}/{repo}')
    
    def get_commits(self, owner, repo, page=1, per_page=30):
        return self._request(f'/repos/{owner}/{repo}/commits', {
            'page': page,
            'per_page': per_page
        })
    


github = GitHubAPITester("")

# %% [markdown]
# Какие поля возвращает API и как они выглядят

result = github.search_repos("language:c++", per_page=100)

if result and 'items' in result:
    print(f"\n Всего найдено: {result['total_count']:,} репозиториев")
    df = pd.DataFrame(result['items'])
    
    print("\n Доступные поля:")
    for col in df.columns:
        print(f"  • {col}")
    
    print("\nПример данных:")
    display(df[['name', 'full_name', 'size', 'forks_count', 'language']])



In [ ]:
# %% [markdown]

# Тестируем разные поисковые запросы
search_queries = [
    "stars:>1000",
    "forks:>500",
    "pushed:>2026-01-01",
    "created:2025-01-01..2025-12-31"
]

for query in search_queries:
    result = github.search_repos(query, per_page=1)
    
    if result:
        total = result['total_count']
        print(f"{query}  {total}")
    
    time.sleep(1)



In [24]:
import base64
import time
import pandas as pd
from datetime import datetime

# %% [markdown]
# ### Задача 1: README с Contributing
# %%
# Соберем две группы проектов:
# 1. Популярные 
# 2. Менее популярные (может не быть Contributing)

#подсчет пулл-реквестов
# Группа A: Очень популярные (5000+ звезд)
query_a = "stars:>5000 language:python"
result_a = github.search_repos(query_a, per_page=10)

# Группа B: Средней популярности (100-500 звезд)
query_b = "stars:100..500 language:python"
result_b = github.search_repos(query_b, per_page=10)

data_h1 = []

# Анализируем первую группу
print("\nзвезд > 5000:")
for repo in result_a['items']:
    owner, name = repo['full_name'].split('/')
    has_contrib = check_contributing(owner, name)
    pr_count = get_pr_count(owner, name)
    
    data_h1.append({
        'repo': repo['full_name'],
        'stars': repo['stargazers_count'],
        'group': 'популярные (>5000)',
        'has_contributing': has_contrib,
        'pr_count': pr_count
    })
    
    print(f"  {repo['full_name']}: Contributing={has_contrib}, PR={pr_count}")
    time.sleep(2)

# Анализируем вторую группу
print("\nзвезд 100-500:")
for repo in result_b['items']:
    owner, name = repo['full_name'].split('/')
    has_contrib = check_contributing(owner, name)
    pr_count = get_pr_count(owner, name)
    
    data_h1.append({
        'repo': repo['full_name'],
        'stars': repo['stargazers_count'],
        'group': 'средние (100-500)',
        'has_contributing': has_contrib,
        'pr_count': pr_count
    })
    
    print(f"  {repo['full_name']}: Contributing={has_contrib}, PR={pr_count}")
    time.sleep(2)

# Анализ результатов
df_h1 = pd.DataFrame(data_h1)

# Сводная таблица
summary = df_h1.groupby(['group', 'has_contributing'])['pr_count'].agg(['count', 'mean'])
print("\nСводка по группам:")
print(summary)



[117] /search/repositories
   Параметры: {'q': 'stars:>5000 language:python', 'page': 1, 'per_page': 10}
   Статус: 200
   Осталось запросов: 28

[118] /search/repositories
   Параметры: {'q': 'stars:100..500 language:python', 'page': 1, 'per_page': 10}
   Статус: 200
   Осталось запросов: 27

звезд > 5000:

[119] /repos/public-apis/public-apis/readme
   Статус: 200
   Осталось запросов: 4822
  public-apis/public-apis: Contributing=True, PR=3983

[120] /repos/EbookFoundation/free-programming-books/readme
   Статус: 200
   Осталось запросов: 4819
  EbookFoundation/free-programming-books: Contributing=True, PR=11564

[121] /repos/donnemartin/system-design-primer/readme
   Статус: 200
   Осталось запросов: 4816
  donnemartin/system-design-primer: Contributing=True, PR=635

[122] /repos/vinta/awesome-python/readme
   Статус: 200
   Осталось запросов: 4813
  vinta/awesome-python: Contributing=True, PR=2229

[123] /repos/TheAlgorithms/Python/readme
   Статус: 200
   Осталось запросов: 4810


In [10]:
# %% [markdown]
# ### Задача 2: Чем больше релизов, тем больше звезд


query = "stars:>1000"
result = github.search_repos(query, per_page=50)  # Возьмем 50 проектов

data_h2 = []

for i, repo in enumerate(result['items']):
    owner, name = repo['full_name'].split('/')
    
    # Получаем релизы
    releases = github._request(f"/repos/{owner}/{name}/releases", {'per_page': 100})
    release_count = len(releases) if releases else 0
    
    # Возраст проекта
    created = datetime.strptime(repo['created_at'], '%Y-%m-%dT%H:%M:%SZ')
    age_months = (datetime.now() - created).days / 30
    
    data_h2.append({
        'repo': repo['full_name'],
        'stars': repo['stargazers_count'],
        'total_releases': release_count,
        'age_months': round(age_months, 1),
        'has_releases': release_count > 0
    })
    
    status = "ЕСТЬ релизы" if release_count > 0 else "нет релизов"
    print(f"{i+1}. {repo['full_name']}: {status} ({release_count})")
    time.sleep(1)
    
df_h2 = pd.DataFrame(data_h2)

# Анализируем только проекты с релизами
df_with_releases = df_h2[df_h2['has_releases'] == True].copy()

print(f"\nНайдено проектов с релизами: {len(df_with_releases)} из {len(df_h2)}")

if len(df_with_releases) > 5:
    corr_total = df_with_releases['stars'].corr(df_with_releases['total_releases'])
    print(f"Корреляция кол-ва звезд и кол-ва релизов: {corr_total:.3f}")
          
    corr_per_month = df_with_releases['stars'].corr(df_with_releases['releases_per_month'])
    print(f"Корреляция кол-ва звезд и кол-ва релизов в месяц): {corr_per_month:.3f}")



[110] /search/repositories
   Параметры: {'q': 'stars:>1000', 'page': 1, 'per_page': 50}
   Статус: 200
   Осталось запросов: 29

[111] /repos/codecrafters-io/build-your-own-x/releases
   Параметры: {'per_page': 100}
   Статус: 200
   Осталось запросов: 4992
1. codecrafters-io/build-your-own-x: нет релизов (0)

[112] /repos/sindresorhus/awesome/releases
   Параметры: {'per_page': 100}
   Статус: 200
   Осталось запросов: 4991
2. sindresorhus/awesome: нет релизов (0)

[113] /repos/freeCodeCamp/freeCodeCamp/releases
   Параметры: {'per_page': 100}
   Статус: 200
   Осталось запросов: 4990
3. freeCodeCamp/freeCodeCamp: нет релизов (0)

[114] /repos/public-apis/public-apis/releases
   Параметры: {'per_page': 100}
   Статус: 200
   Осталось запросов: 4989
4. public-apis/public-apis: нет релизов (0)

[115] /repos/EbookFoundation/free-programming-books/releases
   Параметры: {'per_page': 100}
   Статус: 200
   Осталось запросов: 4988
5. EbookFoundation/free-programming-books: нет релизов (0)


[160] /repos/golang/go/releases
   Параметры: {'per_page': 100}
   Статус: 200
   Осталось запросов: 4943
50. golang/go: нет релизов (0)

Найдено проектов с релизами: 21 из 50
Корреляция кол-ва звезд и кол-ва релизов: -0.480


KeyError: 'releases_per_month'

In [25]:
# %% [markdown]
# ### Задача 3:есть ссылки, значит будет больше человек отслеживать

# %%
social_patterns = ['twitter.com', 'linkedin.com', 'discord', 'slack', 'youtube.com', 't.me']

def check_social_links(owner, repo):
    readme_url = f"/repos/{owner}/{repo}/readme"
    readme_data = github._request(readme_url)
    
    if readme_data and 'content' in readme_data:
        content = base64.b64decode(readme_data['content']).decode('utf-8').lower()
        found = []
        for pattern in social_patterns:
            if pattern in content:
                found.append(pattern)
        return len(found) > 0, found
    return False, []

# Собираем данные
query = "stars:>1000"
result = github.search_repos(query, per_page=20)

data_h3 = []

for i, repo in enumerate(result['items']):
    owner, name = repo['full_name'].split('/')
    
    has_social, socials_found = check_social_links(owner, name)
    
    full_repo = github.get_repo(owner, name)
    watchers = full_repo.get('watchers_count', 0)
    subscribers = full_repo.get('subscribers_count', 0)
    
    data_h3.append({
        'repo': repo['full_name'],
        'stars': repo['stargazers_count'],
        'has_social': has_social,
        'social_count': len(socials_found),
        'socials': ', '.join(socials_found) if socials_found else 'нет',
        'watchers': watchers,
        'subscribers': subscribers
    })
    
    print(f"{i+1}. {repo['full_name']}: соцсети={has_social} ({len(socials_found)}), watchers={watchers}")
    time.sleep(2)

# Статистический анализ
df_h3 = pd.DataFrame(data_h3)


# 1. Сравнение групп
print("\n1. Сравнение проектов с соцсетями и без:")
grouped = df_h3.groupby('has_social')[['watchers', 'subscribers', 'stars']].mean()
print(grouped)

# 2. Процентное соотношение
has_social_pct = df_h3['has_social'].mean() * 100
print(f"\n2. Проектов с соцсетями: {has_social_pct:.1f}%")




[139] /search/repositories
   Параметры: {'q': 'stars:>1000', 'page': 1, 'per_page': 20}
   Статус: 200
   Осталось запросов: 29

[140] /repos/codecrafters-io/build-your-own-x/readme
   Статус: 200
   Осталось запросов: 4762

[141] /repos/codecrafters-io/build-your-own-x
   Статус: 200
   Осталось запросов: 4761
1. codecrafters-io/build-your-own-x: соцсети=True (3), watchers=468294

[142] /repos/sindresorhus/awesome/readme
   Статус: 200
   Осталось запросов: 4760

[143] /repos/sindresorhus/awesome
   Статус: 200
   Осталось запросов: 4759
2. sindresorhus/awesome: соцсети=True (3), watchers=440090

[144] /repos/freeCodeCamp/freeCodeCamp/readme
   Статус: 200
   Осталось запросов: 4758

[145] /repos/freeCodeCamp/freeCodeCamp
   Статус: 200
   Осталось запросов: 4757
3. freeCodeCamp/freeCodeCamp: соцсети=True (2), watchers=437447

[146] /repos/public-apis/public-apis/readme
   Статус: 200
   Осталось запросов: 4756

[147] /repos/public-apis/public-apis
   Статус: 200
   Осталось запросо

In [26]:
# %% [markdown]
# ### Задача 7: Репозитории с permissive-лицензиями (MIT, Apache-2.0) имеют больше stars и forks, чем с другими или без нее

# %%


# Собираем проекты с разными лицензиями
query = "stars:>1000"
result = github.search_repos(query, per_page=50)

data_h7 = []

for i, repo in enumerate(result['items']):
    owner, name = repo['full_name'].split('/')
    
    # Текущая лицензия
    repo_info = github.get_repo(owner, name)
    current_license = repo_info['license']['key'] if repo_info['license'] else None
    
    permissive_licenses = ['mit', 'apache-2.0', 'bsd-3-clause', 'bsd-2-clause']
    if repo['license']:
        license_key = repo_info['license']['key']
        license_name = repo_info['license']['name']
    else:
        license_key = None
        license_name = None
        
    if license_key in permissive_licenses:
        license_type = "permissive"
    elif license_key is None:
        license_type = "no license"
    else:
        license_type = "other"
        
    data_h7.append({
        'repo': repo['full_name'],
        'license_key': license_key,
        'license_name': license_name,
        'license_type': license_type,
        'stars': repo_info['stargazers_count'],
        'forks': repo_info['forks_count'],
        'age_years': round((datetime.now() - datetime.strptime(repo['created_at'], '%Y-%m-%dT%H:%M:%SZ')).days / 365, 1)
    })
    
    print(f"{i+1}. {repo['full_name']}: {license_type} ({license_key}), звезд={repo_info['stargazers_count']}, форков={repo_info['forks_count']}")
    time.sleep(1)

df_h7 = pd.DataFrame(data_h7)

print("\n1. Распеределение проектов по типам лицензий")
license_dist = df_h7['license_type'].value_counts()
for lic_type, count in license_dist.items():
    print(f"{lic_type}: {count} проектов ({count/len(df_h7)*100:.1f}%)")

# 2. Сравнение популярности по типам лицензий
print("\n2.Сравнение популярности оп типам лицензий")

# Группировка по типу лицензии
group_stats = df_h7.groupby('license_type').agg({
    'stars': ['count', 'mean', 'median', 'min', 'max'],
    'forks': ['count', 'mean', 'median']
}).round(0)

print("\nСреднее количество звезд по типам лицензий:")
print(group_stats['stars'])

print("\nСреднее количество форков по типам лицензий:")
print(group_stats['forks'])

        


[301] /search/repositories
   Параметры: {'q': 'stars:>1000', 'page': 1, 'per_page': 50}
   Статус: 200
   Осталось запросов: 29

[302] /repos/codecrafters-io/build-your-own-x
   Статус: 200
   Осталось запросов: 4947
1. codecrafters-io/build-your-own-x: no license (None), звезд=470487, форков=44103

[303] /repos/sindresorhus/awesome
   Статус: 200
   Осталось запросов: 4946
2. sindresorhus/awesome: other (cc0-1.0), звезд=441805, форков=33354

[304] /repos/freeCodeCamp/freeCodeCamp
   Статус: 200
   Осталось запросов: 4945
3. freeCodeCamp/freeCodeCamp: permissive (bsd-3-clause), звезд=437672, форков=43492

[305] /repos/public-apis/public-apis
   Статус: 200
   Осталось запросов: 4944
4. public-apis/public-apis: permissive (mit), звезд=402583, форков=43278

[306] /repos/EbookFoundation/free-programming-books
   Статус: 200
   Осталось запросов: 4943
5. EbookFoundation/free-programming-books: other (cc-by-4.0), звезд=383491, форков=66001

[307] /repos/kamranahmedse/developer-roadmap
   

In [ ]:
# %% [markdown]
# ### Задача 4:Динамика создания новых репозиториев в зависимости от времени года (статистика по месяцам)

def get_season(month):
    if month in [12, 1, 2]:
        return 'Зима'
    elif month in [3, 4, 5]:
        return 'Весна'
    elif month in [6, 7, 8]:
        return 'Лето'
    else: 
        return 'Осень'


months_data = []

test_months = [
    ('2025-03', 'Весна'),
    ('2025-07', 'Лето'),
    ('2025-10', 'Осень'),
    ('2025-12', 'Зима')
]

for year_month, season_name in test_months:
    query = f"created:{year_month}-01..{year_month}-31"
    print(f"\n{season_name} {year_month}:")
    
    result = github.search_repos(query, per_page=1)
    
    if result:
        total = result['total_count']
        print(f"   Создано репозиториев: {total:,}")
        
        months_data.append({
            'month': year_month,
            'season': season_name,
            'total': total
        })
    else:
        print("Ошибка получения данных")
    
    time.sleep(2) 

df_seasons = pd.DataFrame(months_data)


print("\n1. Данные по месяцам:")
print(df_seasons.to_string(index=False))


print("\n2. Сравнение активности летом и осенью:")

summer_data = df_seasons[df_seasons['season'] == 'Лето']
autumn_data = df_seasons[df_seasons['season'] == 'Осень']

if len(summer_data) > 0 and len(autumn_data) > 0:
    summer_avg = summer_data['total'].mean()
    autumn_avg = autumn_data['total'].mean()
    
    print(f"Среднее летом: {summer_avg:.0f} репозиториев")
    print(f"Среднее осенью: {autumn_avg:.0f} репозиториев")
    
    

In [27]:
# %% [markdown]
# ### Задача 8-9: Анализ популярности в зависимости от мультиплатформенности

def project_languages(owner, repo):
    langs_url = f"/repos/{owner}/{repo}/languages"
    langs = github._request(langs_url)
    
    if not langs:
        return None, 0, []
    
    total_bytes = sum(langs.values())
    lang_percent = {}
    
    for lang, bytes_count in langs.items():
        percentage = (bytes_count / total_bytes) * 100
        lang_percent[lang] = round(percentage, 1)
        
    sorted_langs = sorted(lang_percent.items(), key=lambda x: x[1], reverse=True)
    major_langs = [lang for lang, pct in sorted_langs if pct > 20]
    is_multiplatform = len(major_langs) >= 2
    
    return is_multiplatform, len(langs), sorted_langs


query = "stars:>1000"
result = github.search_repos(query, per_page=50)    

data = []

for i, repo in enumerate(result['items']):
    owner, name = repo['full_name'].split('/')
    print(f"\n{i+1}. Анализ {repo['full_name']}...")
    
    repo_info = github.get_repo(owner, name)
    
    is_multiplatform, lang_count, lang_details = project_languages(owner, name)
    
    if is_multiplatform is not None:
        watchers = repo_info.get('watchers_count', 0)
        subscribers = repo_info.get('subscribers_count', 0)
        
        data.append({
            'repo': repo['full_name'],
            'is_multiplatform': is_multiplatform,
            'lang_count': lang_count,
            'main_lang': lang_details[0][0] if lang_details else None,
            'main_lang_pct': lang_details[0][1] if lang_details else 0,
            'second_lang': lang_details[1][0] if len(lang_details) > 1 else None,
            'second_lang_pct': lang_details[1][1] if len(lang_details) > 1 else 0,
            'stars': repo_info['stargazers_count'],
            'forks': repo_info['forks_count'],
            'watchers': watchers,
            'subscribers': subscribers
        })
        
        status = "Мультиплатформенный" if is_multiplatform else "Не мультиплатформенный"
        main = f"{lang_details[0][0]} ({lang_details[0][1]}%)"
        second = f"{lang_details[1][0]} ({lang_details[1][1]}%)" if len(lang_details) > 1 else "нет"
        print(f"   {status}")
        print(f"   Языки: основной={main}, второй={second}")
        print(f"   Звезд={repo_info['stargazers_count']}, Форков={repo_info['forks_count']}")
    
    time.sleep(2)

df = pd.DataFrame(data)

print("\n1. Общая Информация:")
print(f"Всего проектов: {len(df)}")
print(f"Мультиплатформенных: {df['is_multiplatform'].sum()} ({df['is_multiplatform'].mean()*100:.1f}%)")
print(f"Не мультиплатформенных: {len(df) - df['is_multiplatform'].sum()} ({(1-df['is_multiplatform'].mean())*100:.1f}%)")

# 2. Сравнение метрик
print("\n2. СРАВНЕНИЕ МЕТРИК:")

multi = df[df['is_multiplatform'] == True]
single = df[df['is_multiplatform'] == False]

print("\nstars:")
print(f"Мультиплатформенные: среднее={multi['stars'].mean():.0f}, медиана={multi['stars'].median():.0f}")
print(f"Одноязычные: среднее={single['stars'].mean():.0f}, медиана={single['stars'].median():.0f}")
print(f"Разница: {((multi['stars'].mean()/single['stars'].mean())-1)*100:.1f}%")

print("\nforks:")
print(f"Мультиплатформенные: среднее={multi['forks'].mean():.0f}, медиана={multi['forks'].median():.0f}")
print(f"Одноязычные: среднее={single['forks'].mean():.0f}, медиана={single['forks'].median():.0f}")
print(f"Разница: {((multi['forks'].mean()/single['forks'].mean())-1)*100:.1f}%")

print("\nWATCHERS:")
print(f"Мультиплатформенные: среднее={multi['watchers'].mean():.0f}, медиана={multi['watchers'].median():.0f}")
print(f"Одноязычные: среднее={single['watchers'].mean():.0f}, медиана={single['watchers'].median():.0f}")
print(f"Разница: {((multi['watchers'].mean()/single['watchers'].mean())-1)*100:.1f}%")
  
    


[352] /search/repositories
   Параметры: {'q': 'stars:>1000', 'page': 1, 'per_page': 50}
   Статус: 200
   Осталось запросов: 29

1. Анализ codecrafters-io/build-your-own-x...

[353] /repos/codecrafters-io/build-your-own-x
   Статус: 200
   Осталось запросов: 4897

[354] /repos/codecrafters-io/build-your-own-x/languages
   Статус: 200
   Осталось запросов: 4948
   Не мультиплатформенный
   Языки: основной=Markdown (100.0%), второй=нет
   Звезд=470489, Форков=44103

2. Анализ sindresorhus/awesome...

[355] /repos/sindresorhus/awesome
   Статус: 200
   Осталось запросов: 4896

[356] /repos/sindresorhus/awesome/languages
   Статус: 200
   Осталось запросов: 4947

3. Анализ freeCodeCamp/freeCodeCamp...

[357] /repos/freeCodeCamp/freeCodeCamp
   Статус: 200
   Осталось запросов: 4895

[358] /repos/freeCodeCamp/freeCodeCamp/languages
   Статус: 200
   Осталось запросов: 4946
   Не мультиплатформенный
   Языки: основной=TypeScript (76.5%), второй=JavaScript (17.9%)
   Звезд=437672, Форков=43


29. Анализ flutter/flutter...

[409] /repos/flutter/flutter
   Статус: 200
   Осталось запросов: 4869

[410] /repos/flutter/flutter/languages
   Статус: 200
   Осталось запросов: 4920
   Не мультиплатформенный
   Языки: основной=Dart (74.7%), второй=C++ (16.4%)
   Звезд=175399, Форков=30070

30. Анализ twbs/bootstrap...

[411] /repos/twbs/bootstrap
   Статус: 200
   Осталось запросов: 4868

[412] /repos/twbs/bootstrap/languages
   Статус: 200
   Осталось запросов: 4919
   Мультиплатформенный
   Языки: основной=MDX (28.9%), второй=JavaScript (25.8%)
   Звезд=174004, Форков=79070

31. Анализ github/gitignore...

[413] /repos/github/gitignore
   Статус: 200
   Осталось запросов: 4867

[414] /repos/github/gitignore/languages
   Статус: 200
   Осталось запросов: 4918

32. Анализ massgravel/Microsoft-Activation-Scripts...

[415] /repos/massgravel/Microsoft-Activation-Scripts
   Статус: 200
   Осталось запросов: 4866

[416] /repos/massgravel/Microsoft-Activation-Scripts/languages
   Статус: 

In [ ]:
# %% [markdown]
# ### Задача 10: Зависимость срока жизни проекта от количества форков

def lifespan(owner, repo):
    commits_url = f"/repos/{owner}/{repo}/commits"
    
    first_commits = github._request(commits_url, {'per_page': 1,'sort': 'created','direction': 'asc'})
    last_commits = github._request(commits_url, {'per_page': 1,'sort': 'created','direction': 'desc'})
    
    if not first_commits or not last_commits:
        return None
    
    try:
        first_day = datetime.strptime(first_commits[0]['commit']['author']['date'], '%Y-%m-%dT%H:%M:%SZ')
        last_day = datetime.strptime(last_commits[0]['commit']['author']['date'], '%Y-%m-%dT%H:%M:%SZ')
        
        lifetime = (last_day - first_day).days
        return max(lifetime, 1)  # минимум 1 день
    except:
        return None
    
query = "stars:>1000"
result = github.search_repos(query, per_page=50) 
data = []

for i, repo in enumerate(result['items']):
    owner, name = repo['full_name'].split('/')
    print(f"\n{i+1}. Анализ {repo['full_name']}...")
    
    repo_info = github.get_repo(owner, name)
    
    lifetime = lifespan(owner, name)
    
    if lifetime:
        lifetime_years = round(lifetime / 365, 1)
        
        forks = repo_info['forks_count']
        stars = repo_info['stargazers_count']
        
        data.append({
            'repo': repo['full_name'],
            'lifetime_days': lifetime,
            'lifetime_years': lifetime_years,
            'forks': forks,
            'stars': stars,
            'created': repo['created_at'][:10],
            'updated': repo['updated_at'][:10]
        })
        
        print(f"Срок жизни: {lifetime_years} лет ({lifetime} дней)")
        print(f"Форки: {forks}, Звезды: {stars}")
    
    time.sleep(3) 

# Создаем DataFrame
df = pd.DataFrame(data)

# 1. Общая статистика
print("\n1. Общая информация")
print(f"Всего проектов: {len(df)}")
print(f"Средний срок жизни: {df['lifetime_years'].mean():.1f} лет")

print("\n2. Корреляция:")
corr_forks = df['lifetime_years'].corr(df['forks'])
print(f"Корреляция срока жизни и количества форков): {corr_forks:.3f}")

# Корреляция срока жизни и звезд
corr_stars = df['lifetime_years'].corr(df['stars'])
print(f"Корреляция срока жизни и количества звезд): {corr_stars:.3f}")
